In [1]:
import pandas as pd
import numpy as np

In [3]:
df=pd.read_excel("../data/Telco_customer_churn.xlsx")
df.head()

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


In [4]:
print(df.shape)
df.columns.tolist()

(7043, 33)


['CustomerID',
 'Count',
 'Country',
 'State',
 'City',
 'Zip Code',
 'Lat Long',
 'Latitude',
 'Longitude',
 'Gender',
 'Senior Citizen',
 'Partner',
 'Dependents',
 'Tenure Months',
 'Phone Service',
 'Multiple Lines',
 'Internet Service',
 'Online Security',
 'Online Backup',
 'Device Protection',
 'Tech Support',
 'Streaming TV',
 'Streaming Movies',
 'Contract',
 'Paperless Billing',
 'Payment Method',
 'Monthly Charges',
 'Total Charges',
 'Churn Label',
 'Churn Value',
 'Churn Score',
 'CLTV',
 'Churn Reason']

In [5]:
df["Churn Value"].value_counts()

Churn Value
0    5174
1    1869
Name: count, dtype: int64

In [6]:
print(df["Total Charges"].dtype)
df[df["Total Charges"].astype(str).str.strip() == ""].shape

object


(11, 33)

In [7]:
df[df["Total Charges"].astype(str).str.strip() == ""][["CustomerID", "Tenure Months", "Monthly Charges", "Total Charges", "Churn Value"]]

,CustomerID,Tenure Months,Monthly Charges,Total Charges,Churn Value
2234,4472-LVYGI,0,52.55,,0
2438,3115-CZMZD,0,20.25,,0
2568,5709-LVOEQ,0,80.85,,0
2667,4367-NUYAO,0,25.75,,0
2856,1371-DWPAZ,0,56.05,,0
4331,7644-OMVMY,0,19.85,,0
4687,3213-VVOLG,0,25.35,,0
5104,2520-SGTTA,0,20.00,,0
5719,2923-ARZLG,0,19.70,,0
6772,4075-WKNIU,0,73.35,,0


In [8]:
df["Total Charges"] = pd.to_numeric(df["Total Charges"], errors="coerce").fillna(0)
print(df["Total Charges"].dtype)
print(df["Total Charges"].isnull().sum())

float64
0


In [9]:
cols_to_drop = [
    "CustomerID", "Count", "Country", "State", "City", "Zip Code",
    "Lat Long", "Latitude", "Longitude",
    "Churn Label", "Churn Score", "Churn Reason",
]
df_clean = df.drop(columns=cols_to_drop)
print(df_clean.shape)
df_clean.columns.tolist()

(7043, 21)


['Gender',
 'Senior Citizen',
 'Partner',
 'Dependents',
 'Tenure Months',
 'Phone Service',
 'Multiple Lines',
 'Internet Service',
 'Online Security',
 'Online Backup',
 'Device Protection',
 'Tech Support',
 'Streaming TV',
 'Streaming Movies',
 'Contract',
 'Paperless Billing',
 'Payment Method',
 'Monthly Charges',
 'Total Charges',
 'Churn Value',
 'CLTV']

In [10]:
text_cols = df_clean.select_dtypes(include="object").columns
for col in text_cols:
    print(col, "->", df_clean[col].unique().tolist())

Gender -> ['Male', 'Female']
Senior Citizen -> ['No', 'Yes']
Partner -> ['No', 'Yes']
Dependents -> ['No', 'Yes']
Phone Service -> ['Yes', 'No']
Multiple Lines -> ['No', 'Yes', 'No phone service']
Internet Service -> ['DSL', 'Fiber optic', 'No']
Online Security -> ['Yes', 'No', 'No internet service']
Online Backup -> ['Yes', 'No', 'No internet service']
Device Protection -> ['No', 'Yes', 'No internet service']
Tech Support -> ['No', 'Yes', 'No internet service']
Streaming TV -> ['No', 'Yes', 'No internet service']
Streaming Movies -> ['No', 'Yes', 'No internet service']
Contract -> ['Month-to-month', 'Two year', 'One year']
Paperless Billing -> ['Yes', 'No']
Payment Method -> ['Mailed check', 'Electronic check', 'Bank transfer (automatic)', 'Credit card (automatic)']


C:\Users\Administrator\AppData\Local\Temp\ipykernel_15528\1475647045.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_cols = df_clean.select_dtypes(include="object").columns


In [11]:
yes_no_cols = [
    "Senior Citizen", "Partner", "Dependents", "Phone Service", "Paperless Billing",
    "Multiple Lines", "Online Security", "Online Backup", "Device Protection",
    "Tech Support", "Streaming TV", "Streaming Movies",
]

df_clean = df_clean.replace({"No phone service": "No", "No internet service": "No"})

for col in yes_no_cols:
    df_clean[col] = df_clean[col].map({"Yes": 1, "No": 0})

df_clean["Gender"] = df_clean["Gender"].map({"Male": 1, "Female": 0})

print(df_clean[yes_no_cols + ["Gender"]].isnull().sum().sum())

0


In [12]:
df_clean = pd.get_dummies(
    df_clean,
    columns=["Internet Service", "Contract", "Payment Method"],
    drop_first=True,
    dtype=int,
)

print(df_clean.shape)
print(df_clean.dtypes.value_counts())

(7043, 25)
int64      23
float64     2
Name: count, dtype: int64


In [13]:
df_clean.head()

,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Online Security,Online Backup,Device Protection,...,Total Charges,Churn Value,CLTV,Internet Service_Fiber optic,Internet Service_No,Contract_One year,Contract_Two year,Payment Method_Credit card (automatic),Payment Method_Electronic check,Payment Method_Mailed check
0,1,0,0,0,2,1,0,1,1,0,...,108.15,1,3239,0,0,0,0,0,0,1
1,0,0,0,1,2,1,0,0,0,0,...,151.65,1,2701,1,0,0,0,0,1,0
2,0,0,0,1,8,1,1,0,0,1,...,820.50,1,5372,1,0,0,0,0,1,0
3,0,0,1,1,28,1,1,0,0,1,...,3046.05,1,5003,1,0,0,0,0,1,0
4,1,0,0,1,49,1,1,0,1,1,...,5036.30,1,5340,1,0,0,0,0,0,0


In [14]:
from sklearn.model_selection import train_test_split

X = df_clean.drop(columns=["Churn Value"])
y = df_clean["Churn Value"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)
print(y_train.mean().round(3), y_test.mean().round(3))

(5634, 24) (1409, 24)
0.265 0.265


In [15]:
#logistic regression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

log_reg = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)),
])

log_reg.fit(X_train, y_train)

y_pred = log_reg.predict(X_test)
y_prob = log_reg.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 3))

              precision    recall  f1-score   support

           0       0.90      0.73      0.81      1035
           1       0.51      0.79      0.62       374

    accuracy                           0.74      1409
   macro avg       0.71      0.76      0.71      1409
weighted avg       0.80      0.74      0.76      1409

ROC-AUC: 0.849


In [16]:
#random forest
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_rf))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob_rf), 3))

              precision    recall  f1-score   support

           0       0.88      0.82      0.85      1035
           1       0.58      0.68      0.62       374

    accuracy                           0.78      1409
   macro avg       0.73      0.75      0.74      1409
weighted avg       0.80      0.78      0.79      1409

ROC-AUC: 0.843


In [17]:
#gradient boosting
from sklearn.ensemble import HistGradientBoostingClassifier

gb = HistGradientBoostingClassifier(
    class_weight="balanced", random_state=42
)
gb.fit(X_train, y_train)

y_pred_gb = gb.predict(X_test)
y_prob_gb = gb.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_gb))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob_gb), 3))

              precision    recall  f1-score   support

           0       0.90      0.77      0.83      1035
           1       0.54      0.75      0.63       374

    accuracy                           0.77      1409
   macro avg       0.72      0.76      0.73      1409
weighted avg       0.80      0.77      0.78      1409

ROC-AUC: 0.845


In [18]:
print("hello")

hello
